# MultitaskOCTA on Kaggle
Joint retinal-vessel + FAZ segmentation and disease classification (B5 / ILM-OPL, thesis setup).

**Before running:** Settings → Accelerator **GPU T4 x2** (or P100), Internet **On**, and add the two datasets you uploaded
(`multitaskocta_code.zip`, `octa500_b5_subset.zip`, see `kaggle/README.md`). For long runs use *Save Version → Save & Run All (Commit)*.

In [ ]:
# 1. Parameters ---------------------------------------------------------------
FOLDS = [0]                                 # folds to train one after another, e.g. [0, 1, 2, 3, 4] for the full cross-validation
EPOCHS, STARTPOINT = 100, 20                # thesis: 100 epochs, classifier joins after epoch 20
SCRIPT = "faz_vessel_seg_cls.py"            # "classification_baseline.py" = classification-only arm
FOV_MODE = "native"                         # "native" (3 mm + 6 mm scans) or "center3mm" (common field of view, tests the FOV shortcut)
EXTRA = ["--full_eval"]                     # e.g. + ["--scheduler", "onecycle"] or ["--cldice_alpha", "0.8"]
RUN_NAME = "ILM_OPL_large_100const"         # output folder / zip name
LIMIT = 0                                   # >0 builds only the first N scans (smoke test: LIMIT=30, EPOCHS=2, FOLDS=[0])
DEVICE = "cuda"                             # "cpu" for a local dry run

In [ ]:
# 2. Environment ----------------------------------------------------------------
import os, glob, shutil, subprocess, sys, json
INPUT = os.environ.get("KAGGLE_INPUT", "/kaggle/input")
WORK = os.environ.get("KAGGLE_WORKING", "/kaggle/working")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # ONE GPU: DataParallel on 2 GPUs would split the batch of 4 and change BatchNorm statistics
os.environ["WANDB_MODE"] = "disabled"
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

def find(pattern):
    hits = glob.glob(os.path.join(INPUT, "**", pattern), recursive=True)
    assert hits, f"nothing matching {pattern} under {INPUT} - did you add the datasets to the notebook?"
    return hits[0]

code_dir = os.path.join(WORK, "MultitaskOCTA")
shutil.copytree(os.path.dirname(find("faz_vessel_seg_cls.py")), code_dir, dirs_exist_ok=True)
octa500 = os.path.dirname(os.path.dirname(find(os.path.join("Label", "GT_FAZ"))))
print("code:", code_dir, "| raw data:", octa500)

In [ ]:
# 3. Dependencies (torch/torchvision come with the image) ------------------------------
# Installed one by one so that a failure names the package; only the two pinned ones are essential.
reqs = [l.strip() for l in open(os.path.join(code_dir, "requirements-kaggle.txt")) if l.strip() and not l.startswith("#")]
failed = []
for r in reqs:
    res = subprocess.run([sys.executable, "-m", "pip", "install", "-q", r], capture_output=True, text=True)
    print(("OK    " if res.returncode == 0 else "FAILED"), r)
    if res.returncode: failed.append(r); print(res.stderr[-600:])
print("failed:", failed)


In [ ]:
# 4. Build the processed dataset (B5 / ILM_OPL, GT_FAZ + GT_LargeVessel) ----------------
data = os.path.join(WORK, "data", f"ILM_OPL_large_{FOV_MODE}")
cmd = [sys.executable, "Preprocessing/build_processed_octa500.py", "--octa500", octa500, "--out", data, "--projection", "ILM_OPL", "--vessel", "large", "--fov_mode", FOV_MODE]
if LIMIT: cmd += ["--limit", str(LIMIT)]
subprocess.run(cmd, cwd=code_dir, check=True)

In [ ]:
# 5. Train (one fold after the other; per-fold log and timing) -------------------------------
import time
out = os.path.join(WORK, "runs", RUN_NAME)
os.makedirs(os.path.join(out, "summary"), exist_ok=True)
for FOLD in FOLDS:
    cmd = [sys.executable, SCRIPT, "--img_path", os.path.join(data, "image_surface"), "--gt_path", os.path.join(data, "gt_full", "mask"),
           "--save_path", out, "--train_type", "Vessel_FAZ", "--distance_type", "dist_signed11", "--num_epochs", str(EPOCHS), "--startpoint", str(STARTPOINT),
           "--fold", str(FOLD), "--eval_every", "3", "--num_workers", "2", "--device", DEVICE] + EXTRA
    print("fold", FOLD, ":", " ".join(cmd[1:6]), "...", flush=True)
    t0 = time.time()
    with open(os.path.join(out, f"train_fold{FOLD}.log"), "w") as log:
        p = subprocess.Popen(cmd, cwd=code_dir, stdout=log, stderr=subprocess.STDOUT); p.wait()
    print(f"fold {FOLD}: exit code {p.returncode}, {(time.time() - t0) / 60:.1f} min", flush=True)

In [ ]:
# 6. Result per fold: test metrics at the best validation epoch; then mean +- std over the folds -------
subprocess.run([sys.executable, "aggregate_folds.py", out], cwd=code_dir)

In [ ]:
# 7. Package the outputs (download from the Output tab) --------------------------------------
shutil.make_archive(os.path.join(WORK, RUN_NAME), "zip", os.path.join(WORK, "runs"), RUN_NAME)
print(os.listdir(WORK))